# DistilRoBERTa vs TF-IDF: five-seed GPU run (Colab)

This notebook fine-tunes DistilRoBERTa on the five predeclared contract-level splits (seeds 42, 7, 17, 23, 101)
used by the saved TF-IDF results, then compares the two. It only reads `data/cuad/train_separate_questions.json`;
`test.json` is never read.

**Before you start**
1. *Runtime > Change runtime type > T4 GPU*.
2. Add a GitHub token with read access to the repository as a Colab secret named `GITHUB_TOKEN`
   (key icon in the left sidebar, then enable *Notebook access*). Without the secret you are prompted for it.
   The token is sent as an HTTP header and never written to disk or to the clone's git config.
3. Run the cells in order (*Runtime > Run all* works).

**If the session disconnects:** reconnect and *Run all* again. With `USE_DRIVE = True`, results and checkpoints live
on Google Drive: finished seeds are skipped and an interrupted seed resumes from its last saved epoch.

**Runtime (estimate, not a measurement).** The saved TF-IDF reports list 42,382 training chunks across the five seeds
(7,816 to 9,025 per seed). Four epochs means about 170,000 training sequences of up to 512 tokens. Assuming a T4 in
fp16 trains DistilRoBERTa at 40 to 80 sequences a second, training takes about 35 to 70 minutes in total. Scoring is
about 76,000 forward passes (calibration chunks scored once per epoch plus twice at the end, evaluation chunks once),
roughly 5 to 10 minutes. Chunking, installation and cloning add about 15 to 20 minutes. Plan for **about 1 to 1.5
hours**. The log prints the real training time after the first seed, so you can extrapolate from that.

## 1. Settings

In [ ]:
# Settings: the only cell you may need to edit
REPO = "Break-Through-Tech/Accenture-1R-contract-review-challenge"  # GitHub owner/name
BRANCH = "main"                    # must contain scripts/evaluate_encoder_nested.py with --dry-run
SEEDS = [42, 7, 17, 23, 101]       # the predeclared seeds; keep all five for the official comparison
USE_DRIVE = True                   # keep results + checkpoints on Google Drive so a disconnect loses nothing
DRIVE_FOLDER = "/content/drive/MyDrive/clausetriage_encoder"
REPO_DIR = "/content/clausetriage"

## 2. Check the GPU

In [ ]:
import subprocess

import torch

# Fail early with instructions if no CUDA device is attached
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "nvidia-smi not available")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then run all cells again.")
print("GPU:", torch.cuda.get_device_name(0), "| preinstalled torch", torch.__version__)

## 3. Choose where results and checkpoints go

In [ ]:
from pathlib import Path

# Google Drive survives disconnects; the VM's local disk does not
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path(DRIVE_FOLDER)
else:
    BASE = Path("/content/encoder_run")
RESULTS_ROOT = BASE / "results"   # encoder_nested_seed<seed>/ folders, small, meant to be committed
WORK_ROOT = BASE / "work"         # checkpoints (about 1 GB per seed while training), never committed
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_ROOT)
print("checkpoints:", WORK_ROOT)

## 4. Get the repository (private: needs a token)

In [ ]:
import base64
import getpass
import os


def github_token() -> str:
    """Read the token from the GITHUB_TOKEN Colab secret, or prompt for it without echoing."""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            return token
    except Exception:  # secret missing or notebook access not granted
        pass
    return getpass.getpass("GitHub token with read access to the repository: ")


# Clone once per VM; the token travels in an HTTP header, not in the URL or .git/config
if not Path(REPO_DIR, ".git").is_dir():
    auth = base64.b64encode(f"x-access-token:{github_token()}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
    clone = subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                            f"https://github.com/{REPO}.git", REPO_DIR], env=env, capture_output=True, text=True)
    del auth, env
    if clone.returncode != 0:
        raise SystemExit("git clone failed. Check the token's access to the repo and BRANCH.\n" + clone.stderr)

os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
for required in ("scripts/evaluate_encoder_nested.py", "scripts/compare_encoder_tfidf.py",
                 "data/cuad/train_separate_questions.json", "results/word_char_tfidf_nested/report.json"):
    if not Path(required).exists():
        raise SystemExit(f"{required} is missing: is BRANCH right?")

## 5. Install the pinned requirements without replacing Colab's CUDA torch

`requirements.txt` leaves torch unpinned; the constraint file below additionally holds torch at the preinstalled
CUDA build, so pip cannot swap it for a CPU wheel. Training and comparison run in fresh Python processes, so if
Colab offers to restart the session after this cell you can ignore it (if you do restart, run all cells again).

In [ ]:
import sys

constraint = Path("/content/torch-constraint.txt")
constraint.write_text(f"torch=={torch.__version__}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", str(constraint)],
               check=True)

# Confirm, in a fresh process, that torch still sees the GPU and the pinned transformers is in place
check = subprocess.run([sys.executable, "-c",
                        "import torch, transformers; print(torch.__version__, torch.cuda.is_available(), "
                        "transformers.__version__)"], capture_output=True, text=True, check=True)
print(check.stdout)
if " True " not in f" {check.stdout} ":
    raise SystemExit("torch lost CUDA after installing requirements; see the pip output above")

## 6. Download the pinned model files and run the quick checks

Downloads DistilRoBERTa at the pinned revision into the default Hugging Face cache, then runs the unit tests for the
runner and the comparison (about a minute, no training).

In [ ]:
download = (
    "from src.chunking.token_chunker import TOKENIZER_NAME as N, TOKENIZER_REVISION as R\n"
    "from transformers import AutoModel, AutoTokenizer\n"
    "AutoTokenizer.from_pretrained(N, revision=R, use_fast=True)\n"
    "AutoModel.from_pretrained(N, revision=R)\n"
    "print('cached', N, R)"
)
subprocess.run([sys.executable, "-c", download], check=True)
!python -m pytest -q -p no:cacheprovider tests/test_hub.py tests/test_evaluate_encoder_nested.py tests/test_compare_encoder_tfidf.py

## 7. Train and evaluate every seed

For each seed the runner first checks the split against the saved TF-IDF run (contract-ID digests, data hash, chunk
counts, labels) and stops on any mismatch, before any training. Re-running this cell skips finished seeds and resumes an
interrupted one. Progress and per-seed timing are printed as it goes.

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_cmd = (f"python -m scripts.evaluate_encoder_nested --seeds {' '.join(map(str, SEEDS))} "
           f'--results-root "{RESULTS_ROOT}" --work-root "{WORK_ROOT}"')
print(run_cmd)
!{run_cmd}

In [ ]:
import json

# Confirm every seed finished before comparing
status = {}
for seed in SEEDS:
    report = RESULTS_ROOT / f"encoder_nested_seed{seed}" / "report.json"
    status[seed] = json.loads(report.read_text()).get("status") if report.is_file() else "missing"
print(status)
if any(value != "complete" for value in status.values()):
    raise SystemExit("Some seeds are not complete: run the previous cell again (finished seeds are skipped).")

## 8. Compare with the saved TF-IDF runs

In [ ]:
template = f"{RESULTS_ROOT}/encoder_nested_seed" + "{seed}"
compare_cmd = (f'python -m scripts.compare_encoder_tfidf --seeds {" ".join(map(str, SEEDS))} '
               f'--candidate-template "{template}" --output "{RESULTS_ROOT}/encoder_vs_tfidf"')
print(compare_cmd)
!{compare_cmd}

## 9. Package the results for download

In [ ]:
import zipfile

# Only the small result folders: scores, chunk ids, thresholds, metrics, reports. No checkpoints.
bundle = Path("/content/encoder_results.zip")
folders = sorted(RESULTS_ROOT.glob("encoder_nested_seed*")) + [RESULTS_ROOT / "encoder_vs_tfidf"]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in folders:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, Path("results") / path.relative_to(RESULTS_ROOT))
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB) with", [folder.name for folder in folders])

from google.colab import files
files.download(str(bundle))

## 10. Commit the results (on your own machine)

From the repository root, on a new branch:

```sh
git switch -c feature/17-encoder-results
unzip ~/Downloads/encoder_results.zip          # creates results/encoder_nested_seed*/ and results/encoder_vs_tfidf/
python -m scripts.compare_encoder_tfidf        # optional: re-creates results/encoder_vs_tfidf/ locally
git add results/encoder_nested_seed* results/encoder_vs_tfidf
git commit -m "feat(results): add five-seed DistilRoBERTa results and TF-IDF comparison"
```

Do not commit checkpoints (they stay in the Drive `work/` folder, or `outputs/`, which git ignores).
`results/encoder_vs_tfidf/vs_word_char_tfidf/summary.md` holds the verdict; `docs/encoder-comparison.md` explains
how to read it.